# 02 · The analysis frame

Notebook 01 built the 21 predictors for every woman. The later notebooks need a few more
things from the survey file, joined to the same women:

- **the outcome ladder.** ANC1+, ANC4+ and ANC8+ from the same contact count (`m14_1`), so the
  three rungs are cuts through one number and a woman who reaches eight has reached four and one;
- **the survey's own columns** that the maps and the planner describe women by (travel time to a
  health post, listening to the radio, and so on). They describe regions; the models use only
  the 21;
- **the timing of each woman's most recent birth**, for her age at that birth (the one change
  made to the 21: current age is replaced by age at the birth, which is what the pregnancy saw)
  and for the 24-month window the national report uses;
- **the women who started care and when**, for the planner's earlier-first-visit lever.

**Reads:** `data/ETIR8AFL.DTA`, `outputs/ANC8_analysis_frozen.pkl`
**Writes:** `outputs/enhanced/ANC8_analysis_frozen_v2.pkl`, `birth_timing.parquet`, `anc_entry.csv`

In [ ]:
# Settings for this notebook
QUICK = False   # True: fewer resamples for the intervals (a check that everything runs)

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib

from anc8 import cohorts, features
OUT = P.outputs / "enhanced"
OUT.mkdir(parents=True, exist_ok=True)
ir = pd.read_stata(cohorts.IR_FILE, convert_categoricals=False)
first = pd.read_pickle(P.outputs / "ANC8_analysis_frozen.pkl")
first = first[first.anc8plus.notna()].copy()
print(f"{len(first):,} women with a resolved contact count")

In [ ]:
frame, numeric, categorical = features.assemble(first, ir)
frame.to_pickle(OUT / "ANC8_analysis_frozen_v2.pkl")
print(f"analysis frame: {frame.shape[0]:,} women x {frame.shape[1]} columns")
w = frame.v005 / 1e6
for r in lib.RUNGS:
    print(f"  {r}: {np.average(frame[r], weights=w):.1%} of women (survey-weighted)")
timing = cohorts.birth_timing()
print("birth timing for", len(timing), "women")

## The women who started antenatal care, and when

The survey records the timing of the first contact in weeks (`m13a_1` from 100) or months
(from 200). A first contact at 12 weeks or earlier, or in the first three months, is
**early**; later is not; "don't know" (998) and women with no contact are left out. These are
the ANC-entry cohort, on the same split as everyone else.

In [ ]:
t = ir[["caseid", "m13a_1"]].copy()
t["caseid"] = t.caseid.astype(str)
weeks, months = t.m13a_1.between(100, 199), t.m13a_1.between(200, 299)
t["early_anc"] = np.nan
t.loc[weeks, "early_anc"] = (t.m13a_1[weeks] - 100 <= 12).astype(float)
t.loc[months, "early_anc"] = (t.m13a_1[months] - 200 <= 3).astype(float)
cohort = cohorts.load("full")[["caseid", "partition"]]
entry = cohort.merge(t[t.early_anc.notna()], on="caseid", how="inner")
entry = entry.rename(columns={"partition": "entry_partition"})[["caseid", "early_anc", "entry_partition"]]
entry.to_csv(OUT / "anc_entry.csv", index=False)
print(f"{len(entry):,} women started care with a known timing; {entry.early_anc.mean():.0%} of them early (unweighted)")

## Checks against the study

In [ ]:
ref = repro.reference("fingerprints.json")
repro.check("the analysis frame is identical to the study's (content fingerprint)",
            repro.fingerprint(frame) == ref["analysis_frame"], repro.fingerprint(frame))
e = cohorts.entry_table().sort_values("caseid").reset_index(drop=True)
repro.check("the ANC-entry cohort is the study's (content fingerprint)", repro.fingerprint(e) == ref["anc_entry"], f"{len(e):,} women")
repro.report()